# Game W/L Estimator (NBA)
Modelo de clasificación para estimar la probabilidad de victoria entre dos equipos (home vs away) usando datos de *team gamelogs*.
- Target: `WL_NUM` (1=Win, 0=Loss).
- Modelos: Logistic Regression y RandomForestClassifier.
- Métricas: Accuracy, F1, ROC-AUC.
- Visualizaciones: Curva ROC, matriz de confusión, importancia de variables.
- Inferencia “bonita”: parámetro de equipos HOME/AWAY + ventana de últimos N partidos → probabilidad de victoria.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10,6)
plt.rcParams["axes.grid"] = True

## Parámetros de usuario
Sustituye aquí los equipos del partido y el tamaño de la ventana para medias recientes.

In [ ]:
# ←← EDITA AQUÍ
HOME_TEAM = "Boston Celtics"
AWAY_TEAM = "Milwaukee Bucks"
LAST_N = 10  # ventana de forma reciente

SEASON = "2024-25"
PARQUET_PATH = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet

## Carga y preprocesado
- Carga parquet de team gamelogs (dos filas por partido).
- Genera `WL_NUM` si no existe.
- Selecciona solo columnas numéricas útiles (excluye IDs/strings).

In [ ]:
df = pd.read_parquet(PARQUET_PATH)

# WL → numérica
if "WL_NUM" not in df.columns:
    wl_col = df.get("WL")
    df["WL_NUM"] = wl_col.astype(str).str.strip().str.upper().map({"W":1, "L":0})

# Excluir columnas de identificación y obvias no predictoras
exclude_cols = {
    "GAME_ID","GAME_DATE","MATCHUP","TEAM_ID","TEAM_NAME","TEAM_ABBREVIATION",
    "SEASON_YEAR","SEASON_ID","GAMECODE","TEAM_CITY","TEAM_NICKNAME"
}
num_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
# Asegura que el target está presente
num_cols = [c for c in num_cols if c != "WL_NUM"]

# Limpieza simple
df_clean = df[num_cols + ["TEAM_NAME","GAME_DATE","WL_NUM"]].dropna()

## División train/test y modelos
- Train/test = 80/20 con estratificación.
- Baseline (Dummy).
- Logistic Regression (con StandardScaler).
- RandomForestClassifier.
- Métricas: Accuracy, F1, ROC-AUC.

In [ ]:
X = df_clean[num_cols].copy()
y = df_clean["WL_NUM"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_d = dummy.predict(X_test)
y_proba_d = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

# Logistic (pipeline con escalado)
logit = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(max_iter=500))
])
logit.fit(X_train, y_train)
y_pred_l = logit.predict(X_test)
y_proba_l = logit.predict_proba(X_test)[:,1]

# Random Forest
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

def eval_clf(y_true, y_pred, y_proba, name):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    print(f"{name}  →  ACC={acc:.3f} | F1={f1:.3f} | ROC-AUC={auc:.3f}")

print("=== Métricas en test ===")
eval_clf(y_test, y_pred_d, y_proba_d, "Dummy")
eval_clf(y_test, y_pred_l, y_proba_l, "LogReg")
eval_clf(y_test, y_pred_rf, y_proba_rf, "RandomForest")

## Curva ROC y Matriz de confusión (modelo ganador)
Selecciona el mejor modelo por ROC-AUC y grafica.

In [ ]:
# Selección simple por AUC
models = {
    "Dummy": (None, y_pred_d, y_proba_d),
    "LogReg": (logit, y_pred_l, y_proba_l),
    "RandomForest": (rf, y_pred_rf, y_proba_rf)
}
best_name = max(models.items(), key=lambda kv: roc_auc_score(y_test, kv[1][2]))[0]
best_model, best_pred, best_proba = models[best_name]

print(f"Modelo seleccionado: {best_name}")
RocCurveDisplay.from_predictions(y_test, best_proba)
plt.title(f"ROC — {best_name}")
plt.show()

cm = confusion_matrix(y_test, best_pred)
fig, ax = plt.subplots()
im = ax.imshow(cm, cmap="Blues")
ax.set_title("Matriz de confusión")
ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
for (i,j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
plt.show()

## Importancia de variables
- Si el modelo ganador es RF → `feature_importances_`.
- Si es LogReg → magnitud de coeficientes.

In [ ]:
if best_name == "RandomForest":
    importances = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(20)
else:
    # LogReg: coeficientes absolutos normalizados
    coefs = np.abs(best_model.named_steps["clf"].coef_[0])
    importances = pd.Series(coefs, index=X_train.columns).sort_values(ascending=False).head(20)

importances.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top 20 features — {best_name}")
plt.tight_layout()
plt.show()

## Inference “bonita” para HOME vs AWAY
1. Calcula medias de los **últimos N** partidos para HOME y para AWAY (solo columnas `num_cols`).
2. Crea una fila “sintética” representando a HOME con sus medias y a la vez usando columnas `OPP_*` con las medias del rival cuando existan.
   - Ej.: `OPP_EFG_PCT` de HOME ← `EFG_PCT` del AWAY, etc.
3. Pasa esa fila al **modelo ganador** para obtener `P(win_HOME)`.

In [ ]:
def lastN_means(df_team, last_n, cols):
    dft = df_team.sort_values("GAME_DATE").tail(last_n)
    return dft[cols].mean()

# Mapeo simple: si existe "X" y "OPP_X", asigna OPP_X(HOME) = X(AWAY)
def build_match_row(home_stats, away_stats, cols):
    row = home_stats.copy()
    for c in cols:
        if c.startswith("OPP_"):
            base = c.replace("OPP_","")
            if base in away_stats.index:
                row[c] = away_stats[base]
    return row

In [ ]:
home_df = df_clean[df_clean["TEAM_NAME"]==HOME_TEAM]
away_df = df_clean[df_clean["TEAM_NAME"]==AWAY_TEAM]

home_stats = lastN_means(home_df, LAST_N, num_cols)
away_stats = lastN_means(away_df, LAST_N, num_cols)

match_row = build_match_row(home_stats, away_stats, num_cols)
match_row = match_row.reindex(X_train.columns, fill_value=0.0)  # orden/faltantes

proba_home = best_model.predict_proba(match_row.values.reshape(1,-1))[:,1] if best_name!="Dummy" else np.array([0.5])
print(f"{HOME_TEAM} vs {AWAY_TEAM} — P(Home Win) = {proba_home[0]:.3f}")

# Barra “bonita”
plt.figure(figsize=(6,1.2))
plt.barh(["P(Home Win)"], [proba_home[0]])
plt.xlim(0,1); plt.tight_layout(); plt.show()

### Nota
- Esta aproximación usa medias recientes y rellena columnas `OPP_*` con las del rival si existen.
- Si quieres otra ventana (LAST_N), cambia el parámetro.
- Puedes inspeccionar `importances` para crear una selección de features personalizada.